# Gaia colour–magnitude diagram of the Pleiades candidates

This notebook tests the provisional DBSCAN candidates using stellar photometry. Stars in a real open cluster formed at approximately the same time and have similar distances and chemical compositions, so most members should trace a coherent sequence in a colour–magnitude diagram (CMD).

The CMD is an independent scientific validation because photometry was not used by DBSCAN. A clean sequence would support the astrometric selection, while isolated points may indicate binaries, unusual members, measurement problems, or field contamination.

## 1. Imports and project paths

We use NumPy for finite-value checks and the magnitude calculation, and Astropy's `Table` class to preserve the Gaia columns, units, and DBSCAN metadata stored in Notebook 03.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import MaskedColumn, Table

print(f"Python interpreter: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

In [ ]:
project_root = Path.cwd()

if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

candidate_data_path = (
    project_root
    / "data"
    / "processed"
    / "gaia_dr3_pleiades_dbscan_candidates.fits"
)
figures_dir = project_root / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

print(f"Candidate catalogue exists: {candidate_data_path.exists()}")

## 2. Load the provisional candidates

The input is the 1,185-source candidate catalogue created in Notebook 03. We verify that the DBSCAN result columns are present so that core and border candidates can later be compared on the CMD.

In [ ]:
if not candidate_data_path.exists():
    raise FileNotFoundError(
        "The candidate catalogue is missing. Run Notebook 03 first."
    )

gaia_candidates = Table.read(candidate_data_path, format="fits")

required_columns = [
    "source_id",
    "parallax",
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
    "dbscan_label",
    "dbscan_core",
]
missing_columns = [
    name for name in required_columns if name not in gaia_candidates.colnames
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"Candidate rows: {len(gaia_candidates):,}")
print(f"Core candidates: {np.count_nonzero(gaia_candidates['dbscan_core']):,}")
print(f"Border candidates: {np.count_nonzero(~gaia_candidates['dbscan_core']):,}")

## 3. Audit the photometric measurements

A Gaia CMD uses colour $G_{BP}-G_{RP}$ on the horizontal axis and a G-band magnitude on the vertical axis. A source therefore needs finite values for all three Gaia magnitudes. We also need a positive, finite parallax to calculate an absolute magnitude.

As in Notebook 02, the helper below treats both Astropy-masked values and non-finite numerical values as missing. Excluding a source from the CMD does not revoke its astrometric candidate status; it only means this particular validation cannot be performed for that source.

In [ ]:
def missing_value_mask(column):
    """Return True where an Astropy column is masked or non-finite."""
    masked = np.ma.getmaskarray(column)
    values = np.ma.filled(column, np.nan)
    return masked | ~np.isfinite(values)


cmd_input_columns = [
    "phot_g_mean_mag",
    "phot_bp_mean_mag",
    "phot_rp_mean_mag",
    "parallax",
]

photometry_audit = Table(
    rows=[
        (name, np.count_nonzero(missing_value_mask(gaia_candidates[name])))
        for name in cmd_input_columns
    ],
    names=("column", "missing_count"),
)
photometry_audit["missing_percent"] = (
    100 * photometry_audit["missing_count"] / len(gaia_candidates)
)
photometry_audit["missing_percent"].format = ".2f"
photometry_audit

## 4. Create the CMD-ready subset

We combine the finite-value conditions with a positive-parallax requirement. Positive parallax is mathematically necessary for the logarithm in the absolute-magnitude calculation. Every DBSCAN candidate is expected to satisfy it, but the condition is stated explicitly for reproducibility.

The filtered data are copied into `cmd_candidates`; the full `gaia_candidates` table remains unchanged.

In [ ]:
has_cmd_measurements = np.ones(len(gaia_candidates), dtype=bool)

for name in cmd_input_columns:
    has_cmd_measurements &= ~missing_value_mask(gaia_candidates[name])

has_cmd_measurements &= np.asarray(gaia_candidates["parallax"]) > 0
cmd_candidates = gaia_candidates[has_cmd_measurements].copy()

print(f"All astrometric candidates: {len(gaia_candidates):,}")
print(f"CMD-ready candidates: {len(cmd_candidates):,}")
print(f"Unavailable for this CMD: {len(gaia_candidates) - len(cmd_candidates):,}")
print(f"Fraction retained: {len(cmd_candidates) / len(gaia_candidates):.2%}")

## 5. Calculate colour and absolute G magnitude

Gaia colour is the difference between the blue and red photometric bands:

$$G_{BP}-G_{RP}. $$

Smaller values describe bluer, generally hotter stars; larger values describe redder, generally cooler stars.

The distance-modulus relation is $m-M=5\log_{10}(d)-5$, with distance $d$ in parsecs. Using $d=1000/\varpi$ for parallax $\varpi$ in milliarcseconds gives:

$$M_G = G + 5\log_{10}(\varpi)-10. $$

This direct parallax inversion is reasonable as an exploratory calculation for this nearby, tightly selected sample, but it does not incorporate parallax uncertainty or a distance prior. We also have not corrected for interstellar extinction, so the result is an observed absolute magnitude rather than a fully dereddened one.

In [ ]:
cmd_candidates["bp_rp_colour"] = (
    cmd_candidates["phot_bp_mean_mag"]
    - cmd_candidates["phot_rp_mean_mag"]
)
cmd_candidates["absolute_g_mag"] = (
    cmd_candidates["phot_g_mean_mag"]
    + 5 * np.log10(cmd_candidates["parallax"])
    - 10
)

cmd_candidates["bp_rp_colour"].unit = "mag"
cmd_candidates["absolute_g_mag"].unit = "mag"

assert np.all(np.isfinite(cmd_candidates["bp_rp_colour"]))
assert np.all(np.isfinite(cmd_candidates["absolute_g_mag"]))

cmd_candidates[[
    "source_id",
    "bp_rp_colour",
    "absolute_g_mag",
    "dbscan_core",
]][:5]

### Preparation checkpoint

The CMD-ready subset contains the same astrometric candidates except for sources lacking the required photometry. Before drawing conclusions, the next section will plot absolute G magnitude against $G_{BP}-G_{RP}$ and distinguish DBSCAN core from border points.

## 6. Plot the candidate colour–magnitude diagram

A CMD places colour on the horizontal axis and luminosity information on the vertical axis. By astronomical convention, the magnitude axis is inverted: brighter stars have smaller numerical magnitudes and appear near the top.

We plot DBSCAN core points first and border points on top. If both populations trace the same coherent sequence, that supports retaining the border candidates for further assessment. Border points scattered far away from the sequence would be stronger contamination candidates.

The plot initially shows the full finite range rather than imposing a photometric sequence cut.

In [ ]:
cmd_is_core = np.asarray(cmd_candidates["dbscan_core"], dtype=bool)
cmd_is_border = ~cmd_is_core

fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)

ax.scatter(
    cmd_candidates["bp_rp_colour"][cmd_is_core],
    cmd_candidates["absolute_g_mag"][cmd_is_core],
    s=14,
    color="tab:blue",
    alpha=0.65,
    label=f"Core candidates ({np.count_nonzero(cmd_is_core):,})",
)
ax.scatter(
    cmd_candidates["bp_rp_colour"][cmd_is_border],
    cmd_candidates["absolute_g_mag"][cmd_is_border],
    s=38,
    facecolors="none",
    edgecolors="tab:orange",
    linewidths=1.2,
    label=f"Border candidates ({np.count_nonzero(cmd_is_border):,})",
)

ax.set(
    xlabel=r"Gaia colour $G_{BP}-G_{RP}$ (mag)",
    ylabel=r"Absolute G magnitude $M_G$ (mag)",
    title="Gaia CMD of provisional Pleiades candidates",
)
ax.invert_yaxis()
ax.grid(alpha=0.2)
ax.legend()

cmd_figure_path = figures_dir / "04_candidate_colour_magnitude_diagram.png"
fig.savefig(cmd_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"CMD-ready core candidates: {np.count_nonzero(cmd_is_core):,}")
print(f"CMD-ready border candidates: {np.count_nonzero(cmd_is_border):,}")
print(f"Saved figure to: {cmd_figure_path}")

### Questions for interpretation

1. Is there a narrow main sequence running from bright, blue stars toward faint, red stars?
2. Is there a roughly parallel sequence above the main sequence that could contain unresolved binaries, whose combined light makes them brighter?
3. Do the orange border candidates generally follow the stellar sequence, or do they preferentially occupy isolated regions?
4. Are there obvious outliers that warrant checking for photometric-quality problems before calling them contaminants?

The CMD should be used as validation, not as proof that every point on the sequence is a member. Field stars can overlap the cluster sequence, and genuine unusual members can lie away from it.

## 7. Assess BP/RP photometric consistency

Gaia's published `phot_bp_rp_excess_factor`, $C$, compares the combined BP and RP flux with the G-band flux. Its expected value varies strongly with colour, so a single cut on the raw column would be inappropriate.

Following [Riello et al. (2021)](https://arxiv.org/abs/2012.01916), we calculate the colour-corrected factor

$$C^* = C-f(G_{BP}-G_{RP}),$$

where $f$ is their piecewise polynomial. Well-behaved isolated stellar photometry should be centred near $C^*=0$. The paper models its magnitude-dependent scatter as

$$\sigma_{C^*}(G)=0.0059898 + 8.817481\times10^{-12}G^{7.618399}.$$

We use $|C^*|<3\sigma_{C^*}$ as an exploratory photometric-consistency criterion. The authors stress that such filtering must be tailored to the scientific goal: anomalous values reveal inconsistency among the three bands but do not identify which band is responsible, and filtering may also exclude variables or blended systems.

In [ ]:
colour = np.asarray(cmd_candidates["bp_rp_colour"], dtype=float)
raw_flux_excess = np.asarray(
    cmd_candidates["phot_bp_rp_excess_factor"], dtype=float
)
g_magnitude = np.asarray(cmd_candidates["phot_g_mean_mag"], dtype=float)

expected_flux_excess = np.select(
    [colour < 0.5, (colour >= 0.5) & (colour < 4.0), colour >= 4.0],
    [
        1.154360 + 0.033772 * colour + 0.032277 * colour**2,
        (
            1.162004
            + 0.011464 * colour
            + 0.049255 * colour**2
            - 0.005879 * colour**3
        ),
        1.057572 + 0.140537 * colour,
    ],
)

corrected_flux_excess = raw_flux_excess - expected_flux_excess
flux_excess_sigma = (
    0.0059898 + 8.817481e-12 * g_magnitude**7.618399
)
passes_flux_excess = np.abs(corrected_flux_excess) < 3 * flux_excess_sigma

cmd_candidates["corrected_flux_excess"] = corrected_flux_excess
cmd_candidates["passes_flux_excess"] = passes_flux_excess

print(f"Photometrically consistent within 3 sigma: {np.count_nonzero(passes_flux_excess):,}")
print(f"Outside the 3-sigma envelope: {np.count_nonzero(~passes_flux_excess):,}")

In [ ]:
g_grid = np.linspace(g_magnitude.min(), g_magnitude.max(), 400)
sigma_grid = 0.0059898 + 8.817481e-12 * g_grid**7.618399

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.scatter(
    g_magnitude[passes_flux_excess],
    corrected_flux_excess[passes_flux_excess],
    s=10,
    color="tab:blue",
    alpha=0.55,
    label="Within 3 sigma",
)
ax.scatter(
    g_magnitude[~passes_flux_excess],
    corrected_flux_excess[~passes_flux_excess],
    s=16,
    color="tab:red",
    alpha=0.7,
    label="Outside 3 sigma",
)
ax.plot(g_grid, 3 * sigma_grid, color="black", linestyle="--", label=r"$\pm3\sigma_{C^*}$")
ax.plot(g_grid, -3 * sigma_grid, color="black", linestyle="--")
ax.set(
    xlabel="Gaia G magnitude (mag)",
    ylabel=r"Corrected flux excess $C^*$",
    title="BP/RP photometric consistency",
)
ax.legend()
ax.grid(alpha=0.2)

flux_excess_figure_path = figures_dir / "04_corrected_flux_excess.png"
fig.savefig(flux_excess_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {flux_excess_figure_path}")

### Locate the inconsistent measurements on the CMD

We now colour the same CMD by photometric consistency. This does not permanently delete the red points or declare them non-members. It tests whether the scatter that concerned us is associated with the Gaia flux-consistency diagnostic.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
ax.scatter(
    cmd_candidates["bp_rp_colour"][passes_flux_excess],
    cmd_candidates["absolute_g_mag"][passes_flux_excess],
    s=14,
    color="tab:blue",
    alpha=0.65,
    label="Within 3-sigma envelope",
)
ax.scatter(
    cmd_candidates["bp_rp_colour"][~passes_flux_excess],
    cmd_candidates["absolute_g_mag"][~passes_flux_excess],
    s=24,
    color="tab:red",
    alpha=0.75,
    label="Outside 3-sigma envelope",
)
ax.set(
    xlabel=r"Gaia colour $G_{BP}-G_{RP}$ (mag)",
    ylabel=r"Absolute G magnitude $M_G$ (mag)",
    title="CMD coloured by BP/RP consistency",
)
ax.invert_yaxis()
ax.grid(alpha=0.2)
ax.legend()

quality_cmd_figure_path = figures_dir / "04_cmd_photometric_quality.png"
fig.savefig(quality_cmd_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {quality_cmd_figure_path}")

### Interpretation checkpoint

If the red points dominate the broad faint scatter while the blue points form a substantially tighter sequence, much of that scatter is likely photometric rather than evidence against the astrometric selection. The 3-sigma subset is appropriate for displaying a cleaner validation CMD, but the complete candidate catalogue should retain the quality flag so unusual sources are not silently discarded.

## 8. Construct the photometrically consistent validation CMD

We now display only sources inside the 3-sigma flux-excess envelope. This is the cleanest subset for judging whether the astrometric candidates trace a plausible cluster sequence.

First, we compare the pass rates of DBSCAN core and border candidates. A difference may be informative, but it must be interpreted cautiously: border candidates can be fainter on average, and faint BP measurements are intrinsically more difficult. Photometric-quality failure is not equivalent to non-membership.

In [ ]:
quality_by_role = Table(
    names=("dbscan_role", "cmd_ready", "passes_flux_excess", "pass_percent"),
    dtype=("U10", int, int, float),
)

for role_name, role_mask in [
    ("core", cmd_is_core),
    ("border", cmd_is_border),
]:
    role_total = np.count_nonzero(role_mask)
    role_passing = np.count_nonzero(role_mask & passes_flux_excess)
    quality_by_role.add_row(
        (role_name, role_total, role_passing, 100 * role_passing / role_total)
    )

quality_by_role["pass_percent"].format = ".1f"
quality_by_role

In [ ]:
consistent_core = passes_flux_excess & cmd_is_core
consistent_border = passes_flux_excess & cmd_is_border

fig, ax = plt.subplots(figsize=(7, 7), constrained_layout=True)
ax.scatter(
    cmd_candidates["bp_rp_colour"][consistent_core],
    cmd_candidates["absolute_g_mag"][consistent_core],
    s=15,
    color="tab:blue",
    alpha=0.65,
    label=f"Core ({np.count_nonzero(consistent_core):,})",
)
ax.scatter(
    cmd_candidates["bp_rp_colour"][consistent_border],
    cmd_candidates["absolute_g_mag"][consistent_border],
    s=40,
    facecolors="none",
    edgecolors="tab:orange",
    linewidths=1.3,
    label=f"Border ({np.count_nonzero(consistent_border):,})",
)
ax.set(
    xlabel=r"Gaia colour $G_{BP}-G_{RP}$ (mag)",
    ylabel=r"Absolute G magnitude $M_G$ (mag)",
    title="Photometrically consistent Pleiades candidates",
)
ax.invert_yaxis()
ax.grid(alpha=0.2)
ax.legend()

clean_cmd_figure_path = figures_dir / "04_photometrically_consistent_cmd.png"
fig.savefig(clean_cmd_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Photometrically consistent CMD sources: {np.count_nonzero(passes_flux_excess):,}")
print(f"Saved figure to: {clean_cmd_figure_path}")

### Scientific interpretation

The quality-consistent sample should reveal whether the DBSCAN selection contains the expected stellar structure:

- A narrow lower main sequence supports a common distance and age.
- A second ridge up to about 0.75 mag brighter than the single-star sequence is physically plausible for unresolved binaries; an equal-luminosity pair is twice as bright as one component, corresponding to $-2.5\log_{10}(2)\approx-0.75$ mag.
- A small number of isolated points can remain because the flux-excess criterion tests consistency among Gaia bands, not cluster membership.

The sequence provides strong qualitative validation of the astrometric overdensity. It does not by itself yield a membership probability or guarantee that every source is coeval. Before finalising this notebook, we will preserve the CMD quantities and quality flags in a derived catalogue rather than replacing the original DBSCAN candidate file.

## 9. Save the CMD validation results

The output catalogue should preserve all 1,185 astrometric candidates, including the 18 sources without complete CMD photometry. We therefore add:

- `cmd_available`: whether finite G, BP, RP, and positive parallax were available,
- `bp_rp_colour`: $G_{BP}-G_{RP}$ for CMD-ready sources,
- `absolute_g_mag`: the observed absolute G magnitude,
- `corrected_flux_excess`: the Riello et al. corrected $C^*$ value,
- `photometry_consistent_3sigma`: whether $|C^*|<3\sigma_{C^*}$.

The numerical CMD columns are masked for unavailable sources. The Boolean quality result is `False` for those rows, so it must always be interpreted together with `cmd_available`. This creates a new derived file and does not overwrite the Notebook 03 candidate catalogue.

In [ ]:
cmd_validated_candidates = gaia_candidates.copy()
n_candidates = len(cmd_validated_candidates)

cmd_validated_candidates["cmd_available"] = has_cmd_measurements
cmd_validated_candidates["bp_rp_colour"] = MaskedColumn(
    data=np.full(n_candidates, np.nan),
    mask=~has_cmd_measurements,
    unit="mag",
)
cmd_validated_candidates["absolute_g_mag"] = MaskedColumn(
    data=np.full(n_candidates, np.nan),
    mask=~has_cmd_measurements,
    unit="mag",
)
cmd_validated_candidates["corrected_flux_excess"] = MaskedColumn(
    data=np.full(n_candidates, np.nan),
    mask=~has_cmd_measurements,
)
cmd_validated_candidates["photometry_consistent_3sigma"] = np.zeros(
    n_candidates, dtype=bool
)

cmd_validated_candidates["bp_rp_colour"][has_cmd_measurements] = (
    cmd_candidates["bp_rp_colour"]
)
cmd_validated_candidates["absolute_g_mag"][has_cmd_measurements] = (
    cmd_candidates["absolute_g_mag"]
)
cmd_validated_candidates["corrected_flux_excess"][has_cmd_measurements] = (
    corrected_flux_excess
)
cmd_validated_candidates["photometry_consistent_3sigma"][
    has_cmd_measurements
] = passes_flux_excess

cmd_validated_candidates.meta["CSTARSIG"] = 3

print(f"Rows preserved: {len(cmd_validated_candidates):,}")
print(f"CMD available: {np.count_nonzero(cmd_validated_candidates['cmd_available']):,}")
print(
    "Photometrically consistent: "
    f"{np.count_nonzero(cmd_validated_candidates['photometry_consistent_3sigma']):,}"
)

In [ ]:
cmd_validated_data_path = (
    project_root
    / "data"
    / "processed"
    / "gaia_dr3_pleiades_candidates_cmd_validated.fits"
)
cmd_validated_candidates.write(
    cmd_validated_data_path, format="fits", overwrite=True
)

saved_cmd_candidates = Table.read(cmd_validated_data_path, format="fits")

assert len(saved_cmd_candidates) == len(gaia_candidates)
assert len(np.unique(saved_cmd_candidates["source_id"])) == len(saved_cmd_candidates)
assert np.count_nonzero(saved_cmd_candidates["cmd_available"]) == 1_167
assert np.count_nonzero(
    saved_cmd_candidates["photometry_consistent_3sigma"]
) == 1_053
assert np.count_nonzero(
    np.ma.getmaskarray(saved_cmd_candidates["bp_rp_colour"])
) == 18
assert saved_cmd_candidates.meta["CSTARSIG"] == 3

print("Saved CMD-validated catalogue passed all checks.")
print(f"Rows: {len(saved_cmd_candidates):,}")
print(f"Columns: {len(saved_cmd_candidates.colnames)}")
print(f"Saved to: {cmd_validated_data_path}")

## 10. Results and limitations

Of the 1,185 astrometric candidates, 1,167 have the measurements needed for the Gaia CMD and 1,053 also satisfy the exploratory 3-sigma corrected flux-excess criterion. The resulting CMD contains a narrow, physically plausible Pleiades sequence and a brighter parallel population consistent with unresolved binaries. This independently supports the interpretation that DBSCAN identified a real coeval cluster.

Important limitations remain:

- CMD agreement is strong evidence but not a membership probability; field stars can overlap the sequence.
- Sources outside the flux-excess envelope are photometrically inconsistent, not automatically non-members.
- Direct parallax inversion ignores individual distance uncertainty and any parallax zero-point correction.
- No extinction or reddening correction was applied.
- Unresolved binaries, rotation, variability, and stellar activity can broaden or shift the sequence.
- The faint red sequence remains intrinsically harder to validate with Gaia BP photometry.

Notebook 05 will provide another independent assessment by comparing the candidate source identifiers with an appropriate published Pleiades catalogue.